# Faster R-CNN — custom/ON: lặp seeds 7, 21, 123, 2026

Cấu hình chọn bằng validation seed42: **custom_mlp + online jitter 0.1**, best val mAP50–95 **47.6432%**,
cao nhất trong ba cấu hình đã định trước. Giữ các tham số của run5. Mỗi seed khởi động torchrun mới
từ pretrained COCO, có optimizer/RNG/head mới, run Forge và thư mục riêng. Hai T4 dùng DDP cho một seed tại một thời điểm.
Không lấy checkpoint của seed42 hoặc seed trước để train seed sau.

Notebook kiểm cache một lần ở parent; pipeline vẫn kiểm lại cho từng rank/seed.
Mỗi seed xong sẽ tạo ZIP/checksum riêng và cập nhật batch_state.json.
Nếu một seed lỗi, dừng các seed sau, giữ kết quả đã xong và vẫn export logs; batch sẽ báo chưa đủ.
Nếu session bị ngắt, v1 không resume seed đang train. Giữ kết quả đã hoàn thành;
session mới chỉ chọn seed còn thiếu ở cell 5.

Chỉ cấu hình đã chọn được lặp bốn seed, không lặp cả ba cấu hình. Ghép seed42 đã review với bốn seed này thành năm seed.
Mean/std chỉ xuất khi đủ năm seed, dùng **sample std (ddof=1)**. Không chọn seed/model theo test.
Split v1 có frame/cảnh gần nhau, GT val có 39/test có 38 lớp; test đã quan sát.
Không coi đây là blind/unseen-scene benchmark. Chỉ cấu hình đã chọn có năm seed nên không khẳng định
custom/ON luôn tốt hơn các cấu hình còn lại.


## 1. Source và dependencies — giữ như run5


In [ ]:
import os, sys, subprocess, json, time, uuid
from pathlib import Path

REPO = Path('/kaggle/working/VN-Traffic-Sign-Detection')
EXPECTED_SHA = '17ad0eea18df9d414b39748a89eb39d9d06399df'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', 'faster-rcnn',
                    'https://github.com/tuan8p/VN-Traffic-Sign-Detection.git', str(REPO)], check=True)
sha = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
print('Source:', sha)
assert sha == EXPECTED_SHA, 'Source đã thay đổi: đối chiếu commit trước khi chạy tiếp'
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
# Giữ torch/torchvision do Kaggle cung cấp; không upgrade ngầm hoặc cài Pixi win-64 lên Linux.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO),
                'wandb==0.31.0', 'pycocotools==2.0.11'], check=True)

# Ghi cả xung đột package cài sẵn (vd google-adk/OpenTelemetry); không tự hạ package để che cảnh báo.
pip_check = subprocess.run([sys.executable, '-m', 'pip', 'check'], capture_output=True, text=True)
print('pip check exit:', pip_check.returncode)
print(pip_check.stdout or pip_check.stderr)


## 2. GPU, CUDA NMS và mount cache

Nếu runtime khác run5, notebook dừng trước training để đối chiếu; không tự cài lại torch/CUDA.


In [ ]:
import torch, torchvision, pycocotools
from torchvision.ops import nms

print('Python:', sys.version)
print('Torch:', torch.__version__, 'TorchVision:', torchvision.__version__)
print('CUDA:', torch.version.cuda, 'GPUs:', torch.cuda.device_count())
assert torch.cuda.is_available() and torch.cuda.device_count() == 2
assert torch.distributed.is_nccl_available(), 'Runtime chưa có NCCL'
assert hasattr(torch.amp, 'GradScaler'), 'Torch runtime chưa phù hợp API AMP pipeline'
for index in range(2):
    print(index, torch.cuda.get_device_name(index))
    device = f'cuda:{index}'
    kept = nms(torch.tensor([[0., 0., 10., 10.]], device=device),
               torch.tensor([.9], device=device), .5)
    assert kept.cpu().tolist() == [0]
    torch.cuda.synchronize(index)

candidates = sorted(p.parent for p in Path('/kaggle/input').rglob('data.yaml')
                    if p.parent.name == 'processed')
print('Processed roots:', candidates)
assert len(candidates) == 1, 'Chọn đúng một cache processed; không fallback sang raw/local'
DATA_ROOT = candidates[0]


import platform
reference_runtime = {'python': '3.13.15', 'torch': '2.11.0+cu128', 'torchvision': '0.26.0+cu128', 'cuda': '12.8'}
current_runtime = {'python': platform.python_version(), 'torch': str(torch.__version__),
                   'torchvision': str(torchvision.__version__), 'cuda': str(torch.version.cuda)}
print('Reference runtime:', reference_runtime)
print('Current runtime:', current_runtime)
assert current_runtime == reference_runtime, 'Runtime khác run5: dừng đối chiếu trước khi tốn quota'


## 3. Forge login bằng Secret — không tạo run synthetic


In [ ]:
import os
from kaggle_secrets import UserSecretsClient

API_HOST = 'https://api.forge.coreweave.com'
ENTITY = 'HCMUT_IPCV'
PROJECT = 'BTL'

os.environ['WANDB_BASE_URL'] = API_HOST
os.environ['WANDB_MODE'] = 'online'
api_key = UserSecretsClient().get_secret('WANDB_API_KEY')
assert api_key.strip(), 'Kaggle Secret trống'
os.environ['WANDB_API_KEY'] = api_key

import wandb
print('wandb:', wandb.__version__)
assert wandb.login(key=api_key, host=API_HOST, relogin=True, verify=True, timeout=30)
del api_key
print('Đăng nhập thành công; bước tiếp theo sẽ kiểm quyền ghi project.')


## 4. Full-cache preflight dùng chung

Có thể mất 12–14 phút theo các run trước.


In [ ]:
from vn_tsd.config.resolve import resolve_config
from vn_tsd.pipelines.faster_rcnn.config import validate_config
from vn_tsd.pipelines.faster_rcnn.dataset import load_cache

cfg = validate_config(resolve_config(
    pipeline_yaml=REPO/'configs/pipelines/faster_rcnn.yaml',
    shared_yaml=REPO/'configs/shared.yaml', runtime_yaml=REPO/'configs/runtime/kaggle.yaml',
    overrides={'data': {'root': str(DATA_ROOT)}, 'train': {'require_wandb': True}, 'model': {'head': 'custom_mlp'}, 'project': {'wandb_base_url': API_HOST}}))
sets, identity = load_cache(cfg)
meta = json.loads((DATA_ROOT/'meta.json').read_text(encoding='utf-8'))
expected = {'train': (2660, 5743), 'val': (438, 1204), 'test': (371, 829)}
stems = {}
for split, dataset in sets.items():
    info = dataset.identity
    print(split, 'images', info['count'], 'boxes', info['annotations'],
          'small_boxes', info['small_boxes'], 'hash', info['sha256'])
    assert (info['count'], info['annotations']) == expected[split], 'Không khớp snapshotv1'
    assert info['count'] == meta['splits'][split]['n_images']
    assert info['annotations'] == meta['splits'][split]['n_boxes_out']
    listed = [Path(line.strip().replace('\\', '/')).stem for line in
              (DATA_ROOT/f'{split}_files.txt').read_text(encoding='utf-8').splitlines() if line.strip()]
    stems[split] = {r['path'].stem for r in dataset.records}
    assert len(listed) == len(set(listed)) and set(listed) == stems[split]
    assert info['small_boxes'] == 0, 'Báo Bình về box dưới policy8px; không tự lọc'
for a, b in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
    assert not stems[a] & stems[b], f'Trùng stem: {a}/{b}'
for split in ('val', 'test'):
    assert not any('_aug_' in stem for stem in stems[split])
    for stem in stems['train']:
        if '_aug_' in stem:
            assert stem.split('_aug_')[0] not in stems[split], 'Aug parent ở split khác'
seen = {label for r in sets['train'].records for label in r['labels']}
assert seen == set(range(1, 53)), 'Train thiếu lớp'
print('Full cache adapter check passed; chưa chứng minh độc lập cảnh/near-duplicate')


## 5. Khóa cấu hình và tạo batch mới

Mặc định chạy đủ bốn seed. Nếu chạy bù ở session mới, chỉ đổi SEEDS_TO_RUN thành danh sách seed còn thiếu
(ví dụ [123, 2026]); giữ các run cũ để gộp ở local. Không chạy lại seed đã xong để chọn kết quả đẹp.
Chạy lại cell này tạo batch mới; muốn tiếp tục cùng batch trong session hiện tại, không chạy lại cell 5.


In [ ]:
import copy
import shutil
import yaml

ALLOWED_SEEDS = [7, 21, 123, 2026]
SEEDS_TO_RUN = [7, 21, 123, 2026]
assert SEEDS_TO_RUN and len(set(SEEDS_TO_RUN)) == len(SEEDS_TO_RUN)
assert all(type(seed) is int and seed in ALLOWED_SEEDS for seed in SEEDS_TO_RUN)

SELECTED_CONFIG = json.loads('{"project": {"name": "VN-Traffic-Sign-Detection", "course": "IPCV", "wandb_entity": "HCMUT_IPCV", "wandb_project": "BTL", "wandb_base_url": "https://api.forge.coreweave.com", "wandb_run_name": "{pipeline}_s{seed}_ep{epochs}_bs{batch_size}[_online]_{hash8}"}, "seed": 42, "seeds": [42, 7, 21, 123, 2026], "hardware": {"target": "kaggle-2x-t4", "gpus": 2}, "data": {"root": "/kaggle/input/datasets/zero001206/vn-traffic-sign-detection-preprocessed-data/processed", "images_dirname": "images", "labels_dirname": "labels", "class_table": "configs/classes.csv", "num_classes": 52}, "preprocess": {"image_size": [640, 640], "keep_aspect": true, "pad_value": 114, "min_box_px": 8, "hflip": false}, "split": {"use_provided_splits": true, "ratios": {"train": 0.7, "val": 0.15, "test": 0.15}}, "train": {"batch_size": 8, "amp": true, "require_wandb": true, "early_stopping_patience": 10, "epochs": 20, "lr": 0.005, "momentum": 0.9, "weight_decay": 0.0005, "primary_metric": "mAP50-95", "scheduler": "cosine", "warmup_epochs": 1, "warmup_start_factor": 0.001, "min_lr": 1e-06, "run_name": "frcnn_custom_s42_ep20_bs8_on_f616a87c"}, "eval": {"batch_size": 16, "auto_eval_test": true, "split": "test"}, "metrics": {"primary": "mAP50-95", "also": ["mAP50", "mAP75", "AR100"]}, "outputs": {"root": "outputs", "zip_after_train": true}, "online_preprocess": {"enabled": true, "letterbox": true, "normalize": true, "hflip": false, "color_jitter": false, "mosaic": false, "mixup": false, "jitter": 0.1, "mosaic_prob": 1.0, "mixup_prob": 0.1, "mixup_alpha": 32.0, "color_jitter_params": {"brightness": 0.2, "contrast": 0.2, "saturation": 0.2, "hue": 0.02}}, "pipeline": "faster_rcnn", "model": {"family": "faster_rcnn", "backbone": "resnet50", "pretrained": true, "num_classes": 52, "head": "custom_mlp", "head_hidden_dim": 512, "head_dropout": 0.2, "weights": "FasterRCNN_ResNet50_FPN_Weights.COCO_V1", "trainable_backbone_layers": 5, "frozen_bn_eps": 0.0, "score_thresh": 0.001, "nms_thresh": 0.5, "detections_per_img": 100}, "runtime_cfg": {"runtime": "kaggle", "device": "cuda", "num_workers": 2, "pin_memory": true, "gpus": 2, "mixed_precision": true}}')
SEED42_REFERENCE = json.loads('{"seed": 42, "run_name": "frcnn_custom_s42_ep20_bs8_on_f616a87c", "run_id": "k3ogkwzo", "run_url": "https://forge.coreweave.com/wandb/HCMUT_IPCV/BTL/runs/k3ogkwzo", "best_epoch": 16, "best_val": {"mAP50-95": 0.4764324239767427, "mAP50": 0.6448028438299909, "mAP75": 0.5736905349633991, "AR100": 0.6511378101032015}, "test": {"mAP50-95": 0.5041742250180242, "mAP50": 0.6864839503134998, "mAP75": 0.6189286384601914, "AR100": 0.663617817829638}, "source_sha": "17ad0eea18df9d414b39748a89eb39d9d06399df", "config_sha256": "880eda4b80dd17e4a6be8ddf4ea8b65e934a7a27e3d007d4a009ad3a7733143c", "checkpoint_sha256": "99f780f97ee92147a97d4e22d7a3eef89558850f188554b5d2a5ce6fd2544cc9", "dataset_split_hashes": {"train": "fd15fcbf42da08bf274c99b267f1cae512463162a6f0b28ed84d31941c0dbd23", "val": "e5139f467e918e47da0bad10c0068e5bcef193c45ab038bebd0007a475e6d9b3", "test": "3e02b6face375fcbcf441455970e515a4c5350e1638bd76c01ee6d7d62c6d7bb"}, "reference_bundle_sha256": "b52e446c5b69085d93b9e4f735e4d616a78c83844f8e6db9649da04a3eb22297", "scope": "Existing reviewed run5 result; seed42 is not retrained by this notebook."}')
expected_hashes = SEED42_REFERENCE['dataset_split_hashes']
actual_hashes = {s: info['sha256'] for s, info in identity['splits'].items()}
assert actual_hashes == expected_hashes, 'Cache khác run5'
assert sha == SEED42_REFERENCE['source_sha'] == EXPECTED_SHA
assert SELECTED_CONFIG['model']['head'] == 'custom_mlp'
assert SELECTED_CONFIG['online_preprocess']['enabled'] is True
assert SELECTED_CONFIG['online_preprocess']['jitter'] == 0.1

BATCH_NAME = 'frcnn_custom_on_repeats_' + uuid.uuid4().hex[:8]
BATCH_ROOT = Path('/kaggle/working/frcnn_repeats') / BATCH_NAME
BATCH_ROOT.mkdir(parents=True, exist_ok=False)
DOWNLOADS = BATCH_ROOT/'downloads'
DOWNLOADS.mkdir()
STATE = {
    'batch_name': BATCH_NAME, 'status': 'prepared', 'seeds_requested': SEEDS_TO_RUN,
    'selected_configuration': 'custom_mlp / online jitter0.1 / maximum20epochs',
    'selection_rule': 'seed42 validation mAP50-95; not test',
    'source_sha': sha, 'dataset_split_hashes': expected_hashes,
    'runs': {str(seed): {'status': 'pending'} for seed in SEEDS_TO_RUN},
    'supports_training_resume': False,
}
(BATCH_ROOT/'selected_config_seed42.yaml').write_text(yaml.safe_dump(SELECTED_CONFIG, allow_unicode=True), encoding='utf-8')
(BATCH_ROOT/'seed42_reference.json').write_text(json.dumps(SEED42_REFERENCE, indent=2), encoding='utf-8')
(BATCH_ROOT/'cache_preflight.json').write_text(json.dumps(identity, ensure_ascii=False, indent=2), encoding='utf-8')
(BATCH_ROOT/'batch_state.json').write_text(json.dumps(STATE, indent=2), encoding='utf-8')
(BATCH_ROOT/'pip-check.txt').write_text(pip_check.stdout + pip_check.stderr, encoding='utf-8')
with (BATCH_ROOT/'pip-freeze.txt').open('w') as versions:
    subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=versions, check=True)
print('Batch directory:', BATCH_ROOT)
print('Seeds chạy tuần tự:', SEEDS_TO_RUN)
print('Disk free GB:', round(shutil.disk_usage('/kaggle/working').free / 1e9, 2))


## 6. Helpers kiểm config, nghiệm thu từng seed, tracking và export

Không export credentials/netrc hoặc toàn repo.


In [ ]:
import copy, hashlib, math, re
from zipfile import ZipFile, ZIP_STORED
from vn_tsd.pipelines.faster_rcnn.dataset import content_hash
from vn_tsd.pipelines.faster_rcnn.transforms import describe_effective
from vn_tsd.utils.io import load_yaml

def write_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
    temporary.replace(path)

def save_state():
    write_json(BATCH_ROOT/'batch_state.json', STATE)

def file_sha(path):
    with Path(path).open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()

def canonical_config(cfg):
    result = copy.deepcopy(cfg)
    result['seed'] = '<seed>'
    result['data']['root'] = '<same-cache>'
    result['train'].pop('run_name', None)
    return result

def prepare_seed_config(seed):
    overrides = {'seed': seed, 'project': {'wandb_base_url': API_HOST},
        'train': {'epochs': 20, 'require_wandb': True},
        'model': {'pretrained': True, 'head': 'custom_mlp'},
        'online_preprocess': {'enabled': True}, 'eval': {'auto_eval_test': True}}
    cfg = validate_config(resolve_config(
        pipeline_yaml=REPO/'configs/pipelines/faster_rcnn.yaml', shared_yaml=REPO/'configs/shared.yaml',
        runtime_yaml=REPO/'configs/runtime/kaggle.yaml',
        overrides={**overrides, 'data': {'root': str(DATA_ROOT)}}))
    assert canonical_config(cfg) == canonical_config(SELECTED_CONFIG), 'Config drift so với run5'
    assert cfg['seed'] == seed
    online = describe_effective(cfg)['effective']
    assert online['enabled'] is True and online['jitter'] == 0.1
    assert all(not online[k] for k in ('hflip', 'color_jitter', 'mosaic', 'mixup'))
    return overrides

def verify_seed(seed, run_dir):
    metrics = json.loads((run_dir/'metrics.json').read_text())
    history = json.loads((run_dir/'history.json').read_text())
    cfg = load_yaml(run_dir/'resolved_config.yaml')
    source = json.loads((run_dir/'source_identity.json').read_text())
    launch = json.loads((run_dir/'launch_result.json').read_text())
    assert launch['exit_code'] == 0 and metrics['status'] == metrics['test_status'] == 'completed'
    assert metrics['seed'] == cfg['seed'] == seed
    assert canonical_config(cfg) == canonical_config(SELECTED_CONFIG)
    assert source['head'] == EXPECTED_SHA and source['dirty'] is False
    assert metrics['dataset_split_hashes'] == expected_hashes
    assert content_hash(cfg) == metrics['config_sha256']
    assert metrics['effective']['world_size'] == 2 and metrics['effective']['amp'] is True
    online = metrics['effective']['online']['effective']
    assert online['enabled'] is True and online['jitter'] == 0.1
    assert all(not online[k] for k in ('hflip', 'color_jitter', 'mosaic', 'mixup'))
    assert 1 <= len(history) <= 20
    assert [r['epoch'] for r in history] == list(range(1, len(history) + 1))
    assert len(history) == 20 or history[-1]['patience_count'] >= 10
    assert 1 <= metrics['best_epoch'] <= len(history)
    best_row = history[metrics['best_epoch'] - 1]
    assert best_row['best'] is True
    assert abs(best_row['mAP50-95'] - metrics['best_val']['mAP50-95']) < 1e-5
    for row in history:
        assert math.isfinite(row['train_loss']) and math.isfinite(row['val_loss'])
        assert row['successful_updates'] > 0
        assert all(math.isfinite(v) for name in ('train_components', 'val_components') for v in row[name].values())
    assert file_sha(run_dir/'checkpoints/checkpoint_best.pt') == metrics['checkpoint_sha256']
    assert (run_dir/'run.zip').is_file()
    for name in ('best_val', 'test'):
        assert set(metrics[name]) == {'mAP50-95', 'mAP50', 'mAP75', 'AR100'}
        assert all(isinstance(v, (int, float)) and not isinstance(v, bool) and math.isfinite(v) and 0 <= v <= 1
                   for v in metrics[name].values())
    return metrics, len(history), launch

def check_tracking(run_dir, run_name):
    ids = set()
    for path in (run_dir/'wandb_logs').rglob('run-*.wandb'):
        match = re.fullmatch(r'run-([A-Za-z0-9]+)\.wandb', path.name)
        if match:
            ids.add(match.group(1))
    console = (run_dir/'console.log').read_text(encoding='utf-8', errors='replace')
    ids.update(re.findall(re.escape(f'https://forge.coreweave.com/wandb/{ENTITY}/{PROJECT}/runs/') + r'([A-Za-z0-9]+)', console))
    result = {'verification': 'check_ui', 'entity': ENTITY, 'project': PROJECT,
              'training_run_name': run_name, 'lookup_method': 'stable_run_id_from_sdk_or_console'}
    if len(ids) == 1:
        run_id = next(iter(ids))
        result.update(run_id=run_id, run_url=f'https://forge.coreweave.com/wandb/{ENTITY}/{PROJECT}/runs/{run_id}')
        try:
            server_run = wandb.Api(overrides={'base_url': API_HOST}, timeout=30).run(f'{ENTITY}/{PROJECT}/{run_id}')
            assert server_run.id == run_id
            result.update(server_state=server_run.state, display_name=server_run.name,
                          verification='passed' if server_run.state == 'finished' else 'check_ui')
        except Exception as exc:
            result['error_type'] = type(exc).__name__
    else:
        result['candidate_run_ids'] = sorted(ids)
    write_json(run_dir/'tracking_verification.json', result)
    return result

def bundle_seed(run_dir):
    bundle = DOWNLOADS/f'{run_dir.name}-artifacts.zip'
    sidecar = bundle.with_suffix(bundle.suffix + '.sha256')
    if bundle.exists():
        assert sidecar.is_file() and sidecar.read_text().split()[0] == file_sha(bundle)
        return str(bundle)
    temporary = bundle.with_suffix(bundle.suffix + '.partial')
    assert not temporary.exists(), 'Partial export exists: inspect before retrying'
    with ZipFile(temporary, 'x', compression=ZIP_STORED, allowZip64=True) as archive:
        for path in sorted(run_dir.rglob('*')):
            if path.is_file() and path.name != 'run.zip':
                archive.write(path, arcname=f'{run_dir.name}/{path.relative_to(run_dir).as_posix()}')
    with ZipFile(temporary) as archive:
        assert archive.testzip() is None
    temporary.replace(bundle)
    sidecar.write_text(f'{file_sha(bundle)}  {bundle.name}\n', encoding='utf-8')
    return str(bundle)


## 7. Chạy bốn training tuần tự — khoảng bốn giờ dùng GPU

Mỗi seed dùng torchrun hai process và fresh pretrained. Nếu lỗi, dừng các seed sau và ghi failed.
Chạy tiếp cell 8–9 để export kết quả đã có. Khi chạy lại cell 7 cùng batch, seed completed đã verify sẽ được bỏ qua;
seed failed không resume/ghi đè. Tracking check_ui cần xem Forge UI thêm, không tự coi là metrics không tồn tại.


In [ ]:
import traceback

def run_requested_seeds():
    STATE['status'] = 'running'
    save_state()
    for seed in SEEDS_TO_RUN:
        record = STATE['runs'][str(seed)]
        if record['status'] == 'completed':
            run_dir = Path(record['run_dir'])
            verify_seed(seed, run_dir)
            record['bundle'] = bundle_seed(run_dir)
            save_state()
            print(f'Seed {seed}: đã completed, không train lại.')
            continue
        if record['status'] != 'pending':
            STATE['status'] = 'incomplete'
            save_state()
            print(f'Seed {seed}: trạng thái {record["status"]}. Không resume/ghi đè; export và chạy seed này ở batch mới.')
            break
        run_name = f'frcnn_custom_s{seed}_ep20_bs8_on_' + uuid.uuid4().hex[:8]
        run_dir = BATCH_ROOT/'runs'/run_name
        run_dir.mkdir(parents=True, exist_ok=False)
        record.update(status='running', run_name=run_name, run_dir=str(run_dir))
        save_state()
        try:
            overrides = prepare_seed_config(seed)
            overrides_path = run_dir/'launch_overrides.yaml'
            overrides_path.write_text(yaml.safe_dump(overrides), encoding='utf-8')
            write_json(run_dir/'experiment_notes.json', {
                'seed': seed, 'configuration': 'custom_mlp/online jitter0.1',
                'selected_using': 'seed42 validation mAP50-95 among three planned configurations',
                'initialization': 'fresh pretrained COCO_V1; no training resume',
                'split_limitations': 'cachev1 scene/frame overlap; val39/test38 classes with GT',
                'test_exposure': 'test observed previously; not used for configuration or seed selection',
                'reference_seed42_run_id': SEED42_REFERENCE['run_id']})
            command = [sys.executable, '-m', 'torch.distributed.run', '--standalone', '--nproc_per_node=2',
                '-m', 'tools.run_train', '--pipeline', 'faster_rcnn',
                '--config', str(REPO/'configs/pipelines/faster_rcnn.yaml'),
                '--shared', str(REPO/'configs/shared.yaml'),
                '--runtime', str(REPO/'configs/runtime/kaggle.yaml'),
                '--data-root', str(DATA_ROOT), '--overrides', str(overrides_path),
                '--run-name', run_name, '--run-dir', str(run_dir), '--online']
            write_json(run_dir/'launch_command.json', command)
            environment = os.environ.copy()
            assert environment.get('WANDB_API_KEY') and environment.get('WANDB_MODE') == 'online'
            sdk_directory = run_dir/'wandb_logs'
            sdk_directory.mkdir()
            environment['WANDB_DIR'] = str(sdk_directory)
            print(f'\n=== Seed {seed} | {run_name} ===', flush=True)
            started = time.monotonic()
            with (run_dir/'console.log').open('w', encoding='utf-8') as log:
                process = subprocess.Popen(command, cwd=REPO, env=environment, stdout=subprocess.PIPE,
                                           stderr=subprocess.STDOUT, text=True, bufsize=1)
                try:
                    for line in process.stdout:
                        print(line, end='', flush=True)
                        log.write(line)
                        log.flush()
                    exit_code = process.wait()
                except BaseException:
                    if process.poll() is None:
                        process.terminate()
                        try:
                            process.wait(timeout=30)
                        except subprocess.TimeoutExpired:
                            process.kill()
                            process.wait()
                    raise
            wall_seconds = time.monotonic() - started
            write_json(run_dir/'launch_result.json', {'exit_code': exit_code, 'wall_seconds': wall_seconds})
            if exit_code != 0:
                raise RuntimeError(f'Seed {seed}: worker exit {exit_code}; xem traceback đầu tiên trong console.log')
            metrics, epochs, launch = verify_seed(seed, run_dir)
            tracking = check_tracking(run_dir, run_name)
            write_json(run_dir/'parent_acceptance.json', {'status': 'passed', 'seed': seed, 'epochs': epochs,
                'checkpoint_sha256': metrics['checkpoint_sha256'], 'tracking': tracking})
            bundle = bundle_seed(run_dir)
            record.update(status='completed', epochs=epochs, best_epoch=metrics['best_epoch'],
                best_val=metrics['best_val'], test=metrics['test'],
                checkpoint_sha256=metrics['checkpoint_sha256'], config_sha256=metrics['config_sha256'],
                launcher_seconds=launch['wall_seconds'], tracking=tracking, bundle=bundle)
            save_state()
            print(f'Seed {seed} completed; bundle: {bundle}', flush=True)
        except Exception as exc:
            record.update(status='failed', error_type=type(exc).__name__)
            (run_dir/'parent_error.txt').write_text(traceback.format_exc(), encoding='utf-8')
            STATE['status'] = 'incomplete'
            save_state()
            try:
                record['bundle'] = bundle_seed(run_dir)
            except Exception as export_exc:
                record['export_error_type'] = type(export_exc).__name__
            save_state()
            print(f'Seed {seed} FAILED. Các seed sau chưa chạy. Xem logs; tiếp tục cell export.', flush=True)
            break
    else:
        STATE['status'] = 'completed'
        save_state()
    print('Batch status:', STATE['status'])
    print({seed: record['status'] for seed, record in STATE['runs'].items()})

run_requested_seeds()


## 8. Tổng hợp

Seed42 đã chạy ở run5, không train lại. Chỉ xuất mean/std khi đủ bốn seed mới cộng seed42.
Nếu chạy bù một số seed ở batch khác, notebook giữ kết quả từng seed để gộp ở local;
không báo mean/std năm seed khi thiếu. Không bỏ seed điểm thấp.
Std dùng ddof=1, metric dạng fraction; mean_percent và std_percentage_points nhân 100.


In [ ]:
import csv
import statistics

results = [{'seed': 42, 'origin': 'reviewed_run5_reference', 'best_epoch': SEED42_REFERENCE['best_epoch'],
            'best_val': SEED42_REFERENCE['best_val'], 'test': SEED42_REFERENCE['test'],
            'run_id': SEED42_REFERENCE['run_id'], 'run_url': SEED42_REFERENCE['run_url']}]
for seed in SEEDS_TO_RUN:
    record = STATE['runs'][str(seed)]
    if record['status'] == 'completed':
        results.append({'seed': seed, 'origin': 'current_batch', 'best_epoch': record['best_epoch'],
            'best_val': record['best_val'], 'test': record['test'],
            'run_id': record['tracking'].get('run_id'), 'run_url': record['tracking'].get('run_url')})
missing = sorted(set([42, 7, 21, 123, 2026]) - {r['seed'] for r in results})
summary = {'batch_status': STATE['status'], 'complete_five_seeds': not missing, 'missing_seeds': missing,
           'std_definition': 'sample standard deviation, ddof=1; metrics stored as fractions',
           'configuration': 'custom_mlp / jitter0.1 ON', 'results': results}
if not missing:
    summary['aggregate'] = {}
    for split in ('best_val', 'test'):
        summary['aggregate'][split] = {}
        for key in ('mAP50-95', 'mAP50', 'mAP75', 'AR100'):
            values = [r[split][key] for r in results]
            assert len(values) == 5 and all(math.isfinite(v) for v in values)
            summary['aggregate'][split][key] = {'n': 5, 'mean': statistics.mean(values),
                'std_ddof1': statistics.stdev(values), 'mean_percent': 100*statistics.mean(values),
                'std_percentage_points': 100*statistics.stdev(values)}
write_json(BATCH_ROOT/'repeat_summary.json', summary)
columns = ['seed', 'origin', 'best_epoch', 'run_id', 'run_url'] + [
    f'{split}/{key}' for split in ('best_val', 'test') for key in ('mAP50-95', 'mAP50', 'mAP75', 'AR100')]
with (BATCH_ROOT/'seed_metrics.csv').open('w', newline='', encoding='utf-8') as output:
    writer = csv.DictWriter(output, fieldnames=columns)
    writer.writeheader()
    for row in results:
        flat = {key: row[key] for key in ('seed', 'origin', 'best_epoch', 'run_id', 'run_url')}
        flat.update({f'{split}/{key}': value for split in ('best_val', 'test') for key, value in row[split].items()})
        writer.writerow(flat)
print('Đủ 5 seeds:', not missing, '| Còn thiếu:', missing)
if not missing:
    print(json.dumps(summary['aggregate'], indent=2))
else:
    print('Chưa báo mean/std 5 seed; gửi các batch đã xong để gộp khi đủ.')
pending_tracking = [r['seed'] for r in results if r['origin'] == 'current_batch' and
    STATE['runs'][str(r['seed'])]['tracking']['verification'] != 'passed']
print('Seeds cần xác nhận Forge UI thêm:', pending_tracking)


## 9. ZIP tổng để tải về

ZIP tổng chứa từng bundle con, checksum và batch receipts/summary/CSV; không lồng run.zip hoặc raw checkpoint lần nữa.
Artifacts gốc seed42 vẫn ở run5 local, ZIP tổng chỉ giữ reference.
Tải **all-artifacts.zip và .sha256**, giữ Save Version outputs.
Local lưu vào .agents/artifacts/notebooks/run6/ rồi gửi đường dẫn review.
Notebook kết thúc không đồng nghĩa đủ năm seed: kiểm batch_state và repeat_summary.


In [ ]:
MASTER = Path('/kaggle/working') / f'{BATCH_NAME}-all-artifacts.zip'
assert not MASTER.exists(), f'Đã có bundle tổng: {MASTER}'
with ZipFile(MASTER, 'x', compression=ZIP_STORED, allowZip64=True) as archive:
    for path in sorted(BATCH_ROOT.iterdir()):
        if path.is_file():
            archive.write(path, arcname=f'{BATCH_NAME}/{path.name}')
    for path in sorted(DOWNLOADS.iterdir()):
        if path.is_file() and (path.name.endswith('.zip') or path.name.endswith('.sha256')):
            archive.write(path, arcname=f'{BATCH_NAME}/downloads/{path.name}')
with ZipFile(MASTER) as archive:
    assert archive.testzip() is None
master_checksum = MASTER.with_suffix(MASTER.suffix + '.sha256')
master_checksum.write_text(f'{file_sha(MASTER)}  {MASTER.name}\n', encoding='utf-8')
print('Batch status:', STATE['status'])
print('Download:', MASTER)
print('Checksum:', master_checksum)
print('Tải notebook/log, kiểm từng run Forge Finished. Giữ file gốc và gửi artifacts về review.')
